# Africa Flight Price Fine-tuning with QLoRA (adeyemi-kayode)

Fine-tune an **open-source Llama model** using **QLoRA** (Quantized Low-Rank Adaptation) on the **Africa flight prices** dataset to compete with frontier-style behavior on this task.

- **Dataset:** [Karosi/africa-flight-prices](https://huggingface.co/datasets/Karosi/africa-flight-prices) (Hugging Face)
- **Base model:** Llama (e.g. `meta-llama/Llama-3.2-3B`)
- **Method:** QLoRA (4-bit quantization + LoRA adapters)
- **Experiment tracking:** [Weights & Biases](https://wandb.ai/adekayor-karosi)

This extends the **week6** [adeyemi-kayode](https://github.com/adeyemi-kayode) assessment (GPT-4o mini fine-tuning) to **week7**: open-source model fine-tuning with PEFT.

## 1. Install dependencies

In [ ]:
%pip install -qU datasets transformers torch peft bitsandbytes trl accelerate wandb python-dotenv

## 2. Imports and environment

In [ ]:
import os
import torch
from pathlib import Path
from dotenv import load_dotenv
from datasets import load_dataset, Dataset
from huggingface_hub import login
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM

In [ ]:
# Load .env from repo root or current dir
for _path in [Path.cwd() / ".env", Path.cwd().parent.parent.parent / ".env"]:
    if _path.exists():
        load_dotenv(_path, override=True)
        break

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=True)
    print("Hugging Face login OK")
else:
    print("HF_TOKEN not set; ensure you are logged in for gated models (Llama).")

## 3. Load Africa flight price data and prepare for SFT

In [ ]:
HF_DATASET_ID = "Karosi/africa-flight-prices"

dataset = load_dataset(HF_DATASET_ID, split="train")
print(f"Loaded {len(dataset)} examples from {HF_DATASET_ID}")

In [ ]:
def row_to_text(example):
    """Format one example as: Question + Answer: $price (for completion-only loss)."""
    prompt = (
        f"What is the flight price from {example['origin_city']}, {example['origin_country']} "
        f"to {example['destination_city']}, {example['destination_country']}? "
        f"Respond with the price in USD only, no explanation."
    )
    price = float(example["price_usd"])
    # We train only on the part after 'Answer: $' (see DataCollatorForCompletionOnlyLM)
    return prompt + " Answer: $" + f"{price:.2f}"

def add_text_column(ds):
    return ds.add_column("text", [row_to_text(ds[i]) for i in range(len(ds))])

dataset = add_text_column(dataset)
print("Sample:", dataset[0]["text"][:120], "...")

In [ ]:
# Train / validation split (same idea as week6)
split_dataset = dataset.train_test_split(test_size=0.15, seed=42)
train_data = split_dataset["train"]
val_data = split_dataset["test"]
print(f"Train: {len(train_data)}, Validation: {len(val_data)}")

## 4. Weights & Biases (W&B) setup

In [ ]:
import wandb

WANDB_ENTITY = "adekayor-karosi"  # https://wandb.ai/adekayor-karosi
WANDB_PROJECT = "africa-flight-qlora"

wandb_api_key = os.getenv("WANDB_API_KEY")
if wandb_api_key:
    os.environ["WANDB_API_KEY"] = wandb_api_key
    wandb.login()
    os.environ["WANDB_PROJECT"] = WANDB_PROJECT
    os.environ["WANDB_ENTITY"] = WANDB_ENTITY
    LOG_TO_WANDB = True
    print(f"W&B project: {WANDB_ENTITY}/{WANDB_PROJECT}")
else:
    LOG_TO_WANDB = False
    print("WANDB_API_KEY not set; set report_to='none' for this run.")

## 5. Base model and tokenizer

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

## 6. QLoRA: 4-bit quantization config (BitsAndBytes)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.generation_config.pad_token_id = tokenizer.pad_token_id

model = prepare_model_for_kbit_training(model)
print(f"Model memory footprint: {model.get_memory_footprint() / 1e6:.1f} MB")

## 7. LoRA (QLoRA) hyperparameters

In [ ]:
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 8. Completion-only data collator and training config

In [ ]:
RESPONSE_TEMPLATE = " Answer: $"
collator = DataCollatorForCompletionOnlyLM(RESPONSE_TEMPLATE, tokenizer=tokenizer)

In [ ]:
# Training hyperparameters (QLoRA / SFT)
OUTPUT_DIR = "./africa-flight-qlora-output"
MAX_SEQ_LENGTH = 128
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.05
SAVE_STEPS = 25
LOGGING_STEPS = 5
EVAL_STEPS = 25

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=0.3,
    bf16=True,
    fp16=False,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    per_device_eval_batch_size=2,
    report_to="wandb" if LOG_TO_WANDB else "none",
    group_by_length=True,
)

## 9. SFTTrainer and training

In [ ]:
if LOG_TO_WANDB:
    wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="africa-flight-llama-qlora")

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=collator,
)

print("Starting QLoRA fine-tuning...")
trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
if LOG_TO_WANDB:
    wandb.finish()
print(f"Model and tokenizer saved to {OUTPUT_DIR}")

## 10. Inference: test the fine-tuned model

In [ ]:
def ask_flight_price(prompt, model, tokenizer, max_new_tokens=15):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompt = "What is the flight price from Lagos, Nigeria to Nairobi, Kenya? Respond with the price in USD only, no explanation."
full_response = ask_flight_price(prompt, model, tokenizer)
print("Full response:", full_response)
if " Answer: $" in full_response:
    price_part = full_response.split(" Answer: $")[-1].strip()
    print("Extracted price:", price_part)